# **Домашнее задание 1.** Классификация 16S рРНК

Представим, что перед нами стоит задача определения видового состава микроорганизмов в некотором образце. Один из наиболее простых и распространенных подходов к ее решению — секвенирование 16S рРНК (точнее, кодирующего ее гена) и последующее определение таксономической принадлежности полученных последовательностей.

Для проведения такой классификации было разработано большое количество различных методов. В этом домашнем задании мы реализуем упрощенную версию одного из них — **RDP Classifier** ([статья](https://doi.org/10.1128/AEM.00062-07), [GitHub](https://github.com/rdpstaff/classifier), [SourceForge](https://sourceforge.net/projects/rdp-classifier/)). Данный подход использует наивный байесовский классификатор, обученный на **k-мерах** длины 8.

Чтобы лучше понять, как это работает, заметим, что последовательности РНК в некотором смысле тоже являются текстами. Тогда **k-меры** оказываются не более чем символьными **n-граммами**, а значит поверх них мы можем построить уже знакомый нам **Bag-of-Words**. Остальное — лишь дело техники. Давайте приступать.

In [ ]:
from collections.abc import Callable
from typing import Self

import numpy as np
import pandas as pd
from Bio import SeqIO
from numpy.typing import NDArray
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

## 1.1. Токенизация

Начнем с токенизатора для последовательностей РНК.

In [ ]:
class KMerTokenizer:
    def __init__(self, k: int) -> None:
        self.k = k

    def __call__(self, seq: str) -> list[str]:
        ...  # YOUR CODE HERE

In [ ]:
_tokenizer = KMerTokenizer(2)
assert _tokenizer("ACGT") == ["AC", "CG", "GT"]

_tokenizer = KMerTokenizer(3)
assert _tokenizer("ACGTA") == ["ACG", "CGT", "GTA"]

_tokenizer = KMerTokenizer(4)
assert _tokenizer("ACGTAC") == ["ACGT", "CGTA", "GTAC"]

_tokenizer = KMerTokenizer(3)
assert _tokenizer("ACG") == ["ACG"]
assert _tokenizer("AC") == []

## 1.2. Bag-of-Words

Теперь мы готовы написать Bag-of-Words.

In [ ]:
class BagOfWordsVectorizer:
    def __init__(self, tokenizer: Callable[[str], list[str]]) -> None:
        self.tokenizer = tokenizer

    def fit(self, documents: list[str]) -> Self:
        vocabulary: dict[str, int] = {}

        for doc in documents:
            for token in self.tokenizer(doc):
                ...  # YOUR CODE HERE

        self.vocabulary = vocabulary

        return self

    def transform(self, documents: list[str]) -> NDArray[np.int64]:
        if not hasattr(self, "vocabulary"):
            msg = "vectorizer must be fitted before transform."
            raise RuntimeError(msg)

        X = np.zeros((len(documents), len(self.vocabulary)), dtype=np.int64)

        for i, doc in enumerate(documents):
            for token in self.tokenizer(doc):
                ...  # YOUR CODE HERE

        return X

    def fit_transform(self, documents: list[str]) -> NDArray[np.int64]:
        return self.fit(documents).transform(documents)

In [ ]:
_vectorizer = BagOfWordsVectorizer(KMerTokenizer(2))
_X = _vectorizer.fit_transform(["ACGT", "ACAC"])

assert _vectorizer.vocabulary == {"AC": 0, "CG": 1, "GT": 2, "CA": 3}
assert np.array_equal(
    _X,
    np.array(
        [
            [1, 1, 1, 0],
            [2, 0, 0, 1],
        ]
    ),
)
assert np.array_equal(_vectorizer.transform(["ACGA"]), np.array([[1, 1, 0, 0]]))

## 1.3. Naive Bayes classifier

Наконец, мы готовы перейти к реализации **Naive Bayes classifier**. Вспомним, что предсказание модели для документа $x$ имеет вид

$$
\hat y
=
\arg\max_y
\left[
\log P(y)
+
\sum_{j=1}^{M} x_j \log P(v_j \mid y)
\right],
$$

где $x_j$ — количество вхождений токена $v_j \in V$ в данный документ, а $P(v_j \mid y)$ — вероятность этого токена при условии класса $y$.

Остается учесть еще одну небольшую деталь. Если некоторый токен ни разу не встретился в обучающих данных для класса $y$, то оценка его вероятности окажется равна нулю, и при вычислении $\log P(v_j \mid y)$ вселенная схлопнется... а точнее мы получим `np.float64(-inf)`. Чтобы избежать этого, добавим к числу вхождений каждого токена **псевдосчетчик** (pseudocount) $\alpha > 0$. В общем случае такое сглаживание называется **аддитивным сглаживанием** (additive smoothing), или **сглаживанием Лидстоуна** (Lidstone smoothing), а при $\alpha = 1$ — **сглаживанием Лапласа** (Laplace smoothing).

In [ ]:
class NaiveBayesClassifier:
    """
    Обозначения:
        C — количество классов
        V — размер словаря
    """

    def __init__(self, alpha: float = 1.0) -> None:
        self.alpha = alpha

    def fit(self, X: NDArray[np.int64], y: NDArray[np.int64]) -> Self:
        # HINT: используйте `np.unique`
        classes, class_indices, class_counts = ...  # YOUR CODE HERE
        self.classes = classes

        # log P(y), вектор размерности (C,)
        self.log_py = ...  # YOUR CODE HERE

        feature_counts = np.zeros((len(classes), X.shape[1]), dtype=np.float32)
        # HINT: используйте `np.add.at`, чтобы заполнить матрицу
        ...  # YOUR CODE HERE

        feature_counts += self.alpha  # аддитивное сглаживание

        # log P(v|y), матрица размерности (C, V)
        self.log_pvy = ...  # YOUR CODE HERE

        return self

    def predict(self, X: NDArray[np.int64]) -> NDArray[np.int64]:
        # HINT: используйте матричное умножение для эффективного вычисления
        logits = ...  # YOUR CODE HERE
        return self.classes[logits.argmax(axis=1)]

In [ ]:
_X = np.array(
    [
        [2, 0, 0],
        [1, 1, 0],
        [0, 1, 2],
        [0, 0, 2],
    ],
    dtype=np.int64,
)
_y = np.array([-1, -1, 1, 1])

_classifier = NaiveBayesClassifier(alpha=1.0)
_classifier.fit(_X, _y)

assert np.array_equal(_classifier.classes, [-1, 1])
assert np.allclose(_classifier.log_py, np.log([0.5, 0.5]))
assert np.allclose(
    np.exp(_classifier.log_pvy),
    [
        [4 / 7, 2 / 7, 1 / 7],
        [1 / 8, 2 / 8, 5 / 8],
    ],
)

_preds = _classifier.predict(
    np.array(
        [
            [2, 0, 0],
            [0, 0, 2],
        ],
        dtype=np.int64,
    )
)
assert np.array_equal(_preds, [-1, 1])

## 1.4. Общий сбор

Мы почти у цели. Осталось только обучить нашу модель на данных, которые использует настоящий **RDP Classifier**.

In [ ]:
%%bash
mkdir -p data
wget https://zenodo.org/records/10367203/files/RDPClassifier_16S_trainsetNo19_QiimeFormat.zip -P data
unzip -q data/RDPClassifier_16S_trainsetNo19_QiimeFormat.zip -d data

In [ ]:
records = {"id": [], "sequence": []}

for record in SeqIO.parse("data/RDPClassifier_16S_trainsetNo19_QiimeFormat/RefOTUs.fa", "fasta"):
    records["id"].append(record.id)
    records["sequence"].append(str(record.seq).upper())

records = pd.DataFrame(records)
records.head()

In [ ]:
taxonomy = pd.read_csv(
    "data/RDPClassifier_16S_trainsetNo19_QiimeFormat/Ref_taxonomy.txt",
    sep="\t",
    header=None,
    names=["id", "taxonomy"],
)
taxonomy["genus"] = taxonomy["taxonomy"].str.extract(r"g__([^;]+)", expand=False)
taxonomy.head()

In [ ]:
taxonomy["genus"].value_counts()

Немного упростим себе задачу и оставим только рода, в которых не менее 50 представителей.

In [ ]:
taxonomy = taxonomy[taxonomy.groupby("genus").transform("size") >= 50]

In [ ]:
data = records.merge(taxonomy, on="id", how="inner")
data = data.drop_duplicates(["sequence", "genus"])
data.head()

In [ ]:
len(data)

Разобьем данные на обучающую и тестовую подвыборки.

In [ ]:
train, test = train_test_split(data, test_size=0.2, stratify=data["genus"])
len(train), len(test)

Построим Bag-of-Words на k-мерах длины 8.

In [ ]:
vectorizer = ...  # YOUR CODE HERE

X_train = ...  # YOUR CODE HERE
X_test = ...  # YOUR CODE HERE

Какой размер словаря вы ожидали бы получить? Предположите, почему в реальности он оказался больше.

In [ ]:
len(vectorizer.vocabulary)

Обучим классификатор.

In [ ]:
classifier = ...  # YOUR CODE HERE
...  # YOUR CODE HERE

Сделаем предсказания и посчитаем метрики.

In [ ]:
preds = ...  # YOUR CODE HERE
print(classification_report(test["genus"], preds))

In [ ]:
preds

Предположите, чем могут быть обусловлены столь высокие значения метрик.